In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from __future__ import annotations

In [ ]:
import anndata as ad
from adjustText import adjust_text

from cellassign import assign_cats

from cellbender.remove_background.downstream import load_anndata_from_input_and_output as load_anndata_cellbender

import cellrank as cr
from cellrank.estimators import GPCCA

import doubletdetection

from fa2 import ForceAtlas2

import gc

import harmonypy as hm

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import font_manager, rcParams

import networkx as nx

import numpy as np

import palantir

import pandas as pd

import phate

import plotly.express as px

from pybiomart import Server

import re 

from rpy2.robjects import globalenv
from rpy2.robjects import pandas2ri

import scanpy as sc
import scanpy.external as sce

import scFates as scf

from scib_metrics.benchmark import Benchmarker, BioConservation, BatchCorrection

import scipy.sparse as sp
from scipy.sparse import csr_matrix, issparse

import scvelo as scv

import seaborn as sns

from sklearn.decomposition import PCA

import triku as tk
import os, subprocess

In [ ]:
import sys

sys.path.append('..')

from pyfuncs.io import save_deg_to_excel_simple, load_full_adata, add_ensembl_ids
from pyfuncs.dropletQC import classify_empty_and_damaged
from pyfuncs.general import preprocessing_adata_sub
from pyfuncs.plot_functions import magma, set_plotting_style, plot_volcano, plot_cell_stats, plot_gene_stats
from pyfuncs.common_vars import BASE_DIR, SEED, CELLBENDER_FIXED_ARGS
set_plotting_style()

In [ ]:
from pyfuncs.qc import  MT_CONTIG_MOUSE_REFSEQ, compute_qc_metrics, add_droplet_qc, flag_doublets, qc_embedding, plot_qc_overview, nf_band_report, ambient_top_genes, apply_qc_flags, qc_summary
from pyfuncs.normalization import concat_samples, preliminary_clusters, scran_size_factors, apply_size_factors, compare_normalizations, size_factor_report
from pyfuncs.processing import select_hvgs_preliminary, build_embeddings, select_hvgs_triku, soup_vs_hvg_report, harmony_merge_report
from pyfuncs.characterization import subset_and_reprocess, check_marker_dict, population_composition
from pyfuncs.cell_types import DICT_MARKERS_MAJOR_POPULATIONS, DICT_MARKERS_FAP, DICT_MARKERS_KRANOCYTE, DICT_MARKERS_SATELLITE, DICT_MARKERS_TENO, DICT_RENAMING, PALETTE_CELL_TYPE
from pyfuncs.plot_functions import savefig

In [ ]:
from datetime import date
TODAY = str(date.today())

DATA_DIR = f"{BASE_DIR}/data/ARAUZO_03/"
FIG_DIR = f"{BASE_DIR}/figures/{TODAY}/"
RESULTS_DIR = f"{BASE_DIR}/results/{TODAY}/"
os.makedirs(RESULTS_DIR, exist_ok=True)

CELL_TYPE="cell_type"
SEED = 10

In [ ]:
# Save excel file
def build_deg_df(
    adata,
    group_key='cell_type',
    method='wilcoxon',
    n_genes=None,
    reference='rest',
    use_raw=False,
    reuse_if_present=True
):
    """
    Crea un DataFrame con DEGs por población:
      columns: population, gene, lfc, pvals_adj, pvals_adjxlfc
      orden: por population y pvals_adjxlfc descendente
    """
 
    rgg = adata.uns['rank_genes_groups']
    groups = rgg['names'].dtype.names  # nombres de grupos/poblaciones

    rows = []
    for g in groups:
        names = np.array(rgg['names'][g], dtype=str)
        lfc   = np.array(rgg['logfoldchanges'][g], dtype=float)
        padj  = np.array(rgg['pvals_adj'][g], dtype=float)

        # Evita inf por padj=0 y asegura rango (0,1]
        padj_safe = np.clip(padj, 1e-1000, 1.0)
        score = lfc * (-np.log10(padj_safe))  # pvals_adjxlfc

        rows.append(
            pd.DataFrame({
                'population': g,
                'gene': names,
                'lfc': lfc,
                'pvals_adj': padj,
                'pvals_adjxlfc': score
            })
        )

    df = pd.concat(rows, ignore_index=True)
    # Limpieza y orden
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df.dropna(subset=['lfc','pvals_adj','pvals_adjxlfc'], inplace=True)
    df.sort_values(['population','pvals_adjxlfc'], ascending=[True, False], inplace=True)
    df.reset_index(drop=True, inplace=True)
    return df



# Data loading

In [ ]:
adata = sc.read(f"{DATA_DIR}/processed_adatas/AA_inhouse_dataset_processed.h5ad")
adata_FAP = sc.read(f"{DATA_DIR}/processed_adatas/AA_inhouse_dataset_processed_FAPs.h5ad")

In [ ]:
sc.tl.rank_genes_groups(adata, groupby=CELL_TYPE)

In [ ]:
df_degs = build_deg_df(adata, group_key="cell_type", )
df_degs

In [ ]:
save_deg_to_excel_simple(df_degs, f"{RESULTS_DIR}/1AA_df_degs.xlsx")

## Volcano plots

In [ ]:
for population in df_degs["population"].unique():
    print(population)
    plot_volcano(adata, population, plot_positive_only=True, bottomn=0)

## Dotplots + UMAPs

### Selected markers

In [ ]:
fig, ax = plt.subplots(1, 1)
sc.pl.dotplot(adata=adata, groupby=CELL_TYPE, var_names=["Sema3c", "Smoc2", "Mgp", "Fbln1"], ax=ax)
savefig(fig=fig, filename="1AA_dotplot_selected-markers_all-cell-types", fig_dir=FIG_DIR)

In [ ]:
fig = sc.pl.dotplot(adata=adata_FAP, groupby=CELL_TYPE, var_names=["Sema3c", "Smoc2", "Mgp", "Fbln1"], return_fig=True)
savefig(fig=fig, filename="1AA_dotplot_selected-markers_FAPs", fig_dir=FIG_DIR)

In [ ]:
fig = sc.pl.umap(adata_FAP, color=[CELL_TYPE, "Sema3c", "Smoc2", "Mgp", "Fbln1"], cmap=magma, frameon=False, ncols=3, return_fig=True)
plt.tight_layout()
savefig(fig=fig, filename="1AA_UMAP_selected-markers_FAPs", fig_dir=FIG_DIR)

In [ ]:
fig = sc.pl.umap(adata, color=[CELL_TYPE, "Sema3c", "Smoc2", "Mgp", "Fbln1"], cmap=magma, frameon=False, ncols=3, return_fig=True)
savefig(fig=fig, filename="1AA_UMAP_selected-markers_all-cell-types", fig_dir=FIG_DIR)

### General markers

In [ ]:
DICT_PLOTTING_MARKERS = {"ENDO":  ["Cd36", "Fabp4", "Cd93", "Podxl", "Tie1"],
                         "FAP.1": ["Adamts16", "Rorb", "Wnt10b", "Sbsn", "Uchl1"], 
                         "FAP.2": ['Meox1', 'Clu', 'Daam2', 'Etl4', 'Clec1a' ], 
                         "FAP.3": ["Gdf10", "C7", "C2", "C4b", "Serpina3n"], 
                         "FAP.4": ["Prex2", "Colgalt2", "Hunk", "Lrrtm3", "Cldn15"], 
                         "FAP.5": ["Smim41", "Saa1", "Shisa3", "Gfra2", "Thrsp"], 
                         "FAP.6": ["Tenm2", "Col28a1", "Tec", "Foxd1", "Foxs1"], 
                         "SAT":   ["Pax7", "Fgfr4", "Myf5", "Edn3", "Gal"], 
                         "TNMD":  ["Fmod", "Cilp2", "Chad", "Scx", "Tnc"], }

In [ ]:
fig = sc.pl.dotplot(adata=adata, groupby=CELL_TYPE, var_names=[elemento for valores in DICT_PLOTTING_MARKERS.values() for elemento in valores], vmax=2.5, return_fig=True)
savefig(fig=fig, filename="1AA_dotplot_DEGs_all-cell-types", fig_dir=FIG_DIR)

In [ ]:
fig = sc.pl.dotplot(adata=adata_FAP, groupby=CELL_TYPE, var_names=[elemento for (key, valores) in DICT_PLOTTING_MARKERS.items() for elemento in valores if "FAP" in key ], vmax=2, return_fig=True)
savefig(fig=fig, filename="1AA_dotplot_DEGs_FAPs", fig_dir=FIG_DIR)

### Cdd10 + Cxcl14 + Sox9

In [ ]:
IMF_MARKERS = ["Mme", "Cxcl14", "Sox9"]

In [ ]:
sc.tl.score_genes(adata_FAP, gene_list=IMF_MARKERS, score_name="score_imf")

In [ ]:
fig = sc.pl.umap(adata_FAP, color=[CELL_TYPE] + IMF_MARKERS + ["score_imf"], cmap=magma, frameon=False, ncols=3, return_fig=True)
plt.tight_layout()
savefig(fig=fig, filename="1AA_UMAP_imf-markers_FAPs", fig_dir=FIG_DIR)

## Gene ontology of cell types

In [ ]:
CELL_TYPE

In [ ]:
from pyfuncs.ontology import build_background, background_sensitivity, top_degs, run_goea, collapse_redundant, plot_goea

In [ ]:
fondos = {
    "global":   build_background(adata, min_cells=10),
    "por_pop":  build_background(adata, min_cells=10, group_key=CELL_TYPE),
    "hvg":      build_background(adata, use_hvg=True),
}

degs = top_degs(adata, "cell_type", n_top=200, recompute=True)

rob = background_sensitivity(degs, fondos, )

In [ ]:
res = run_goea(degs, fondos["por_pop"])
res["log_padj"] = -np.log10(res["padj_global"])

In [ ]:
fig, ax = plot_goea(res, n_top=8)
savefig(fig=fig, filename="1AA_GO-terms_top-shared", fig_dir=FIG_DIR)

In [ ]:
for poblacion in res["poblacion"].unique():
    df_sub = res[res["poblacion"] == poblacion]
    fig, ax = plt.subplots(1, 1, figsize=(10, 5))
    sns.barplot(data = df_sub.iloc[:20], y="termino", x="log_padj", ax=ax)
    plt.axvline(-np.log10(0.05), c="#bc0000")
    plt.title(f"Top GO terms for {poblacion}")
    savefig(fig=fig, filename=f"1AA_GO-terms_{poblacion}", fig_dir=FIG_DIR)
    plt.show()


In [ ]:
save_deg_to_excel_simple(res, f"{RESULTS_DIR}/1AA_df_GOEA.xlsx", group_col='poblacion',
    cols=('libreria','termino','solapamiento','odds_ratio', "pval", "padj_global", "genes"))